In [1]:
import json
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

# Input and output files
INPUT_PATH = Path("/content/full_DVSK_data_aiteamvn_256.json")
OUTPUT_PATH = Path("/content/input_chunks.json")

# Selection settings
PAGE_START = 1
PAGE_END = 739
GROUP_SIZE = 3
MAX_PER_GROUP = 2
MIN_TOKENS = 100

COMMENTARY_PATTERNS = [
    r"^\s*Sử thần\b",
    r"^\s*Sử thần Ngô Sĩ Liên\b",
    r"^\s*Ngô Sĩ Liên nói\b",
    r"^\s*Lê Văn Hưu nói\b",
    r"^\s*Lời bàn\b",
]


def load_json(path: Path) -> list[dict]:
    """Load a JSON file containing a list of chunks."""
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, list):
        raise ValueError("Input must be a JSON list.")

    return data


def save_json(data, path: Path) -> None:
    """Save data to a formatted UTF-8 JSON file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)


def get_pages(chunk: dict) -> list[int]:
    """Return valid page numbers as a sorted integer list."""
    pages = chunk.get("pages", [])

    if not isinstance(pages, list):
        return []

    result = []

    for page in pages:
        try:
            result.append(int(page))
        except (TypeError, ValueError):
            continue

    return sorted(set(result))


def get_start_page(chunk: dict) -> int | None:
    """Return the first page of a chunk."""
    pages = get_pages(chunk)
    return pages[0] if pages else None


def get_token_count(chunk: dict) -> int:
    """Return token_count as an integer."""
    try:
        return int(chunk.get("token_count", 0))
    except (TypeError, ValueError):
        return 0


def starts_with_commentary(text: str) -> bool:
    """Check whether the text starts with historical commentary."""
    return any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in COMMENTARY_PATTERNS
    )


def validate_chunk(chunk: dict) -> tuple[bool, str]:
    """Apply basic quality filters before random sampling."""
    text = str(chunk.get("raw_text") or "").strip()
    start_page = get_start_page(chunk)

    if not chunk.get("chunk_id"):
        return False, "missing_chunk_id"

    if not text:
        return False, "empty_raw_text"

    if start_page is None or not PAGE_START <= start_page <= PAGE_END:
        return False, "invalid_pages"

    if get_token_count(chunk) < MIN_TOKENS:
        return False, "token_count_too_low"

    if starts_with_commentary(text):
        return False, "starts_with_commentary"

    return True, "valid"


def get_group_id(page: int) -> int:
    """Assign a page to a consecutive three-page group."""
    return (page - PAGE_START) // GROUP_SIZE + 1


def select_chunks(chunks: list[dict]) -> tuple[list[dict], dict]:
    """Randomly select up to two valid chunks from each page group."""
    groups = defaultdict(list)
    rejected = Counter()

    for chunk in chunks:
        valid, reason = validate_chunk(chunk)

        if not valid:
            rejected[reason] += 1
            continue

        group_id = get_group_id(get_start_page(chunk))
        groups[group_id].append(chunk)

    rng = random.SystemRandom()
    selected = []

    total_groups = (PAGE_END - PAGE_START + GROUP_SIZE) // GROUP_SIZE

    for group_id in range(1, total_groups + 1):
        candidates = groups.get(group_id, [])
        selected_count = min(MAX_PER_GROUP, len(candidates))
        chosen = rng.sample(candidates, selected_count)

        selected.extend(chosen)

    return selected


def main() -> None:
    """Run random selection and save the selected chunks and report."""
    chunks = load_json(INPUT_PATH)
    selected = select_chunks(chunks)

    save_json(selected, OUTPUT_PATH)

    print(f"Input chunks    : {len(chunks)}")
    print(f"Selected chunks : {len(selected)}")
    print(f"Output          : {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Input chunks    : 5343
Selected chunks : 494
Output          : /content/input_chunks.json
